In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from duckduckgo_search import DDGS

query = "econometrics machine learning research"

results = []
links = []

# 1️⃣ SEARCH THE WEB
with DDGS() as ddgs:
    search_results = ddgs.text(query, max_results=100)

    for r in search_results:
        links.append(r["href"])


# 2️⃣ VISIT EACH PAGE AND SCRAPE
for url in links:

    try:
        response = requests.get(
            url,
            headers={"User-Agent": "Mozilla/5.0"},
            timeout=10
        )

        soup = BeautifulSoup(response.text, "html.parser")

        # page title
        title = soup.title.text if soup.title else ""

        # extract visible text
        paragraphs = soup.find_all("p")
        text = " ".join([p.get_text() for p in paragraphs])

        results.append({
            "url": url,
            "title": title,
            "text": text[:5000]   # limit text length
        })

        print("Scraped:", url)

    except Exception as e:
        print("Failed:", url)


# 3️⃣ BUILD DATASET
df = pd.DataFrame(results)

print(df.head())


# 4️⃣ SAVE DATASET
df.to_csv("web_scraped_dataset.csv", index=False)